# **Raw Data Preparation for Pre-train the BERT Model**

#### **Import Libraries**

In [63]:
import torch.nn as nn
from torchtext.vocab import build_vocab_from_iterator
from torchtext.data.utils import get_tokenizer
from transformers import BertTokenizer
from torchtext.datasets import IMDB
import random
import torch

In [2]:
tokenizor = BertTokenizer.from_pretrained("bert-base-uncased")

In [11]:
text = "transformers are powerful models. I am playing cricket"
tokens = tokenizor.tokenize(text)
tokens

['transformers',
 'are',
 'powerful',
 'models',
 '.',
 'i',
 'am',
 'playing',
 'cricket']

In [16]:
#Dataset unpack
train_iter, val_iter = IMDB()
train_iter

ShardingFilterIterDataPipe

In [17]:
def yield_tokens(data_iter):
    for label, text in data_iter:
        yield tokenizor.tokenize(text)

In [62]:
PAD_IDX, CLS_IDX, SEP_IDX, MASK_IDX, UNK_IDX = 0,1,2,3,4
special_tokens = ['PAD_IDX','CLS_IDX', 'SEP_IDX', 'MASK_IDX', 'UNK_IDX']


vocab = build_vocab_from_iterator(yield_tokens(train_iter),
                                  specials= special_tokens,
                                  special_first=True, max_tokens=1000)
vocab.set_default_index(UNK_IDX)

vocab_size = len(vocab)

In [60]:
vocab(tokens)
vocab.get_itos()[53]


'there'

### **Text Masking** 

In [22]:
def bernoulli_true_false (percentage):
    number:bool = random.random() < percentage
    return number
    

In [61]:
def index_to_text(index):
    return vocab.get_itos()[index]

In [ ]:
def text_masking(token):
    mask = bernoulli_true_false(0.2)
    
    if not mask:
        return token, '[PAD]' # if mask false then return as it is and the label also
    
    random_opp = bernoulli_true_false(0.5)
    random_switch = bernoulli_true_false(0.5)
    
    # case 1: if mask is true, random_opp is true and random_sqitch is true
    if mask and random_opp and random_switch:
        token_ = '[MASK]'
        mask_label =  index_to_text(torch.randint(0, vocab_size,(1,)))
    
    
    